<a href="https://colab.research.google.com/github/tu25002-4828-alt/EU_M_Math-Repository/blob/main/Chap08_Cm_01_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import numpy as np
import numpy.random as random
import scipy as sp
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
%matplotlib inline

import sklearn

%precision 3

'%.3f'

In [24]:
import requests, zipfile
import io

url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.data'
res = requests.get(url).content


auto = pd.read_csv(io.StringIO(res.decode('utf-8')), header=None)
auto.columns = ['symboling', 'normalized-losses','make','fuel-type','aspiration','num-of-doors',
                'bodystyle','drive-wheels','engine-location','wheel-base', 'length','width', 'height',
                'curb-weight','engine-type','num-of-cylinders','engine-size','fuel-system',
                'bore','stroke','compression-ratio','horsepower','peak-rpm','city-mpg','highway-mpg',
                'price']

In [25]:
print('自動車の形式：{}'.format(auto.shape))

自動車の形式：(205, 26)


In [26]:
auto.head()
# 'engine-size' を追加
auto = auto[['price', 'horsepower', 'width', 'height', 'engine-size']]
auto.isin(['?']).sum()
auto = auto.replace('?', np.nan).dropna()
print('自動車データの形式：{}'.format(auto.shape))

自動車データの形式：(199, 5)


In [27]:
print('データ型の確認(型変換前)\n{}\n'.format(auto.dtypes))

データ型の確認(型変換前)
price           object
horsepower      object
width          float64
height         float64
engine-size      int64
dtype: object



In [28]:
auto = auto.assign(price=pd.to_numeric(auto.price))
auto = auto.assign(horsepower=pd.to_numeric(auto.horsepower))
print('データ型の確認(型変換後)\n{}'.format(auto.dtypes))

データ型の確認(型変換後)
price            int64
horsepower       int64
width          float64
height         float64
engine-size      int64
dtype: object


In [29]:
auto.corr()

,price,horsepower,width,height,engine-size
price,1.000000,0.810533,0.753871,0.134990,0.873887
horsepower,0.810533,1.000000,0.615315,-0.087407,0.822713
width,0.753871,0.615315,1.000000,0.309223,0.729466
height,0.134990,-0.087407,0.309223,1.000000,0.075569
engine-size,0.873887,0.822713,0.729466,0.075569,1.000000


In [30]:
#8-1
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# 説明変数に width と engine-size を指定
X = auto[['width', 'engine-size']]
y = auto['price']

X = X.astype(float)
y = y.astype(float)

# データを分割 (random_state=0)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=0
)

# モデル作成
model = LinearRegression()

# 学習
model.fit(X_train, y_train)

# スコア表示
print('train score: {:.3f}'.format(model.score(X_train, y_train)))
print('test score: {:.3f}'.format(model.score(X_test, y_test)))

train score: 0.762
test score: 0.839


In [31]:
# 8-4
from sklearn.linear_model import Lasso

lasso_model = Lasso(alpha=10.0, max_iter=2000, random_state=0)

lasso_model.fit(X_train, y_train)

print('Lasso (alpha=10.0) - train score: {:.3f}'.format(lasso_model.score(X_train, y_train)))
print('Lasso (alpha=10.0) - test score: {:.3f}'.format(lasso_model.score(X_test, y_test)))

print('\n--- 各説明変数の係数 ---')
for col, coef in zip(X.columns, lasso_model.coef_):
    print('{}: {:.3f}'.format(col, coef))

Lasso (alpha=10.0) - train score: 0.762
Lasso (alpha=10.0) - test score: 0.839

--- 各説明変数の係数 ---
width: 870.114
engine-size: 128.504


正則化の目的は主に2つある。

訓練データに対してモデルを適合させすぎると，ノイズまで学習してする恐れがある。これにより，テストデータに対する予測精度が落ちてしまう懸念がある。正則化項は，モデルの係数が大きくなりすぎることにペナルティを課すことで，過学習を防ぐ役割があると学んだ。

また，説明変数同士に強い相関がある場合，通常の重回帰では係数の分散が非常に大きくなるため推計値が不安定になる。正則化を導入することで，この多重共線性による悪影響を抑え、安定した推計を可能とする。



ラッソ回帰のメリット
重要度の低い変数の係数を完全に0に落とす性質があるので，大量に変数を投入しても「どの変数が予測に本当に必要か」を自動的に選択してくれる。これによってモデルの解釈性が大幅に向上すると考えられる。
ただし，多重共線性のある変数が複数ある場合、その中からランダムに1つだけを選び，残りを0にしてしまう傾向がある。

リッジ回帰のメリット
相関の高い変数が複数あっても、どれか1つを削るのではなく、関係するすべての変数の係数を一様に小さく調整する。そのため，多重共線性が疑われるデータ群に対して，最も安定した予測性能を発揮します。
係数が完全に0になることはないため，変数が多い場合はモデルの解釈が少し複雑になりがちとなる。